In [ ]:
# ## Cell 1 — Install Libraries & Imports
# Install required libraries (run once per Colab session)
# !pip install yfinance pandas-datareader requests --quiet

import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
import requests
from datetime import datetime
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Establish global timeframe constraints
START = "2005-01-01"
END   = "2024-01-01"   # inclusive through 2023-12-31

print("✅ Libraries loaded. START =", START, "| END =", END)

✅ Libraries loaded. START = 2005-01-01 | END = 2024-01-01


In [36]:
# ## Cell 2 — FRED Helper Function
# FRED is the primary free source for developed-market bond yields.
# Yields are daily (weekdays). Series list: https://fred.stlouisfed.org/

def get_fred(series_id, label=None):
    """Download a FRED series and return a labelled DataFrame."""
    label = label or series_id
    df = web.DataReader(series_id, "fred", start=START, end=END)
    df.columns = [label]
    print(f"  ✅ FRED {series_id}: {len(df)} obs | {df.index[0].date()} → {df.index[-1].date()}")
    return df

# Quick test — US 2Y and 10Y
dgs2  = get_fred("DGS2",  "US_2Y_Yield")
dgs10 = get_fred("DGS10", "US_10Y_Yield")
dgs2.tail(3)

  ✅ FRED DGS2: 4956 obs | 2005-01-03 → 2024-01-01
  ✅ FRED DGS10: 4956 obs | 2005-01-03 → 2024-01-01


,US_2Y_Yield
DATE,
2023-12-28,4.26
2023-12-29,4.23
2024-01-01,NaN


In [37]:
# ## Cell 3 — Yahoo Finance Helper Function
# Used for major equity indices globally and 2-Year sovereign yield fallbacks.

def get_yf(ticker, label=None, col="Close"):
    """Download Yahoo Finance ticker and return a labelled Close-price series."""
    label = label or ticker

    # Download data silently without progress bars to keep Colab clean
    df = yf.download(ticker, start=START, end=END, auto_adjust=False, progress=False)

    if df.empty:
        print(f"  ⚠️  Yahoo Finance: no data for {ticker}")
        return pd.DataFrame(columns=[label])

    # yfinance sometimes returns a DataFrame for a single column. This catches it.
    s = df[col]
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]

    s = s.rename(label)
    print(f"  ✅ YF {ticker}: {len(s)} obs | {s.index[0].date()} → {s.index[-1].date()}")
    return s.to_frame()

# Quick test — S&P 500
sp500 = get_yf("^GSPC", "SP500")
sp500.tail(3)

  ✅ YF ^GSPC: 4781 obs | 2005-01-03 → 2023-12-29


,SP500
Date,
2023-12-27,4781.580078
2023-12-28,4783.350098
2023-12-29,4769.830078


In [38]:
# ## Cell 4 — United States
# Fetch and combine US baseline variables: 2Y Yield, 10Y Yield, and S&P 500

us_2y  = get_fred("DGS2",  "US_2Y")
us_10y = get_fred("DGS10", "US_10Y")
us_eq  = get_yf("^GSPC",   "US_SP500")

# Merge into a single DataFrame aligned strictly by Date
us_df = us_2y.join(us_10y, how="outer").join(us_eq, how="outer")
us_df.index.name = "Date"

print("\n── United States Data Preview ──")
print(us_df.tail(3))

  ✅ FRED DGS2: 4956 obs | 2005-01-03 → 2024-01-01
  ✅ FRED DGS10: 4956 obs | 2005-01-03 → 2024-01-01
  ✅ YF ^GSPC: 4781 obs | 2005-01-03 → 2023-12-29

── United States Data Preview ──
            US_2Y  US_10Y     US_SP500
Date                                  
2023-12-28   4.26    3.84  4783.350098
2023-12-29   4.23    3.88  4769.830078
2024-01-01    NaN     NaN          NaN


In [39]:
# ## Cell 5 — Eurozone Countries (FRED OECD + Yahoo Finance)
# Pulls monthly 10-Year baseline yields from FRED and daily equity from YF.
# Note: Daily Eurozone sovereign yields are pulled later via the ECB API.

oecd_10y_map = {
    "DE": ("Germany",     "Deutsche Bundesbank",        "IRLTLT01DEM156N", "^GDAXI",   "DE_DAX"),
    "FR": ("France",      "Bank of France",             "IRLTLT01FRM156N", "^FCHI",    "FR_CAC40"),
    "IT": ("Italy",       "Bank of Italy",              "IRLTLT01ITM156N", "FTSEMIB.MI","IT_FTSEMIB"),
    "ES": ("Spain",       "Bank of Spain",              "IRLTLT01ESM156N", "^IBEX",    "ES_IBEX35"),
    "NL": ("Netherlands", "De Nederlandsche Bank",      "IRLTLT01NLM156N", "^AEX",     "NL_AEX"),
    "BE": ("Belgium",     "National Bank of Belgium",   "IRLTLT01BEM156N", "^BFX",     "BE_BEL20"),
    "AT": ("Austria",     "Austrian National Bank",     "IRLTLT01ATM156N", "^ATX",     "AT_ATX"),
    "FI": ("Finland",     "Bank of Finland",            "IRLTLT01FIM156N", "^OMXH25",  "FI_OMX25"),
    "GR": ("Greece",      "Bank of Greece",             "IRLTLT01GRM156N", "GD.AT",    "GR_ASE"),
    "IE": ("Ireland",     "Central Bank of Ireland",    "IRLTLT01IEM156N", "^ISEQ",    "IE_ISEQ"),
    "PT": ("Portugal",    "Bank of Portugal",           "IRLTLT01PTM156N", "^PSI20",   "PT_PSI20"),
}

ez_10y_frames = {}
ez_eq_frames  = {}

for iso, (country, bank, fred_id, eq_tk, eq_lbl) in oecd_10y_map.items():
    print(f"\n── {country} ──")

    # 1. Fetch 10-Year Yield
    try:
        ez_10y_frames[iso] = get_fred(fred_id, f"{iso}_10Y")
    except Exception as e:
        print(f"  ❌ FRED {fred_id}: {e}")

    # 2. Fetch Equity Index
    try:
        ez_eq_frames[iso] = get_yf(eq_tk, eq_lbl)
    except Exception as e:
        print(f"  ❌ YF {eq_tk}: {e}")

print("\n✅ Eurozone 10Y and equity done.")


── Germany ──
  ✅ FRED IRLTLT01DEM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^GDAXI: 4824 obs | 2005-01-03 → 2023-12-29

── France ──
  ✅ FRED IRLTLT01FRM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^FCHI: 4859 obs | 2005-01-03 → 2023-12-29

── Italy ──
  ✅ FRED IRLTLT01ITM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF FTSEMIB.MI: 4830 obs | 2005-01-03 → 2023-12-29

── Spain ──
  ✅ FRED IRLTLT01ESM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^IBEX: 4852 obs | 2005-01-03 → 2023-12-29

── Netherlands ──
  ✅ FRED IRLTLT01NLM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^AEX: 4862 obs | 2005-01-03 → 2023-12-29

── Belgium ──
  ✅ FRED IRLTLT01BEM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^BFX: 4859 obs | 2005-01-03 → 2023-12-29

── Austria ──
  ✅ FRED IRLTLT01ATM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^ATX: 4722 obs | 2005-01-03 → 2023-12-29

── Finland ──
  ✅ FRED IRLTLT01FIM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^OMXH25: 2708 obs | 2013-03-05 → 2023-12-29

── 

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^PSI20']: YFPricesMissingError('possibly delisted; no price data found  (1d 2005-01-01 -> 2024-01-01)')


  ✅ FRED IRLTLT01PTM156N: 229 obs | 2005-01-01 → 2024-01-01
  ⚠️  Yahoo Finance: no data for ^PSI20

✅ Eurozone 10Y and equity done.


In [40]:
# ## Cell 6 — ECB Statistical Data Warehouse (Daily Bond Yields)
# ECB SDW provides daily Euro Area benchmark bond yields.
# Dataset: YC | Series: AAA Euro area government bonds (Svensson model)

import requests
from io import StringIO
import pandas as pd

def get_ecb_sdw(dataset, key, label):
    """Fetch daily data from ECB API and return a labelled DataFrame."""
    url = (
        f"https://data-api.ecb.europa.eu/service/data/{dataset}/{key}"
        f"?format=csvdata&startPeriod={START[:10]}&endPeriod={END[:10]}"
    )
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            print(f"  ❌ ECB SDW {key}: HTTP {r.status_code}")
            return pd.DataFrame(columns=[label])

        df = pd.read_csv(StringIO(r.text))

        if "TIME_PERIOD" not in df.columns or "OBS_VALUE" not in df.columns:
            print(f"  ❌ ECB SDW {key}: unexpected columns")
            return pd.DataFrame(columns=[label])

        df["Date"] = pd.to_datetime(df["TIME_PERIOD"])
        df = df.set_index("Date")[["OBS_VALUE"]].rename(columns={"OBS_VALUE": label})
        df = df.dropna()

        print(f"  ✅ ECB SDW {key}: {len(df)} obs | {df.index[0].date()} → {df.index[-1].date()}")
        return df

    except Exception as e:
        print(f"  ❌ ECB SDW API Error: {e}")
        return pd.DataFrame(columns=[label])

print("── Euro Area AAA Government Bond Yield Curve (Daily) ──")
# 2-Year Spot Rate
ecb_2y  = get_ecb_sdw("YC", "B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y",  "EA_2Y_ECB")
# 10-Year Spot Rate
ecb_10y = get_ecb_sdw("YC", "B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y", "EA_10Y_ECB")

if not ecb_2y.empty and not ecb_10y.empty:
    print("\n── ECB Data Preview ──")
    ecb_preview = ecb_2y.join(ecb_10y, how="outer")
    print(ecb_preview.tail(3))

── Euro Area AAA Government Bond Yield Curve (Daily) ──
  ✅ ECB SDW B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y: 4856 obs | 2005-01-03 → 2023-12-29
  ✅ ECB SDW B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y: 4856 obs | 2005-01-03 → 2023-12-29

── ECB Data Preview ──
            EA_2Y_ECB  EA_10Y_ECB
Date                             
2023-12-27   2.424210    1.987609
2023-12-28   2.414324    2.002842
2023-12-29   2.436346    2.080886


In [41]:
# ## Cell 7 — Other Developed Markets (FRED OECD + Yahoo Finance)
# Non-Eurozone developed markets.
# Pulls monthly 10-Year baselines from FRED and daily equity from YF.

other_dev_map = {
    "GB": ("United Kingdom", "IRLTLT01GBM156N", "^FTSE",   "UK_FTSE100"),
    "CH": ("Switzerland",    "IRLTLT01CHM156N", "^SSMI",   "CH_SMI"),
    "SE": ("Sweden",         "IRLTLT01SEM156N", "^OMX",    "SE_OMX30"),
    "NO": ("Norway",         "IRLTLT01NOM156N", "^OBX",    "NO_OBX"),
    "DK": ("Denmark",        "IRLTLT01DKM156N", "^OMXC25", "DK_OMXC25"),
    "JP": ("Japan",          "IRLTLT01JPM156N", "^N225",   "JP_Nikkei225"),
    "AU": ("Australia",      "IRLTLT01AUM156N", "^AXJO",   "AU_ASX200"),
    "NZ": ("New Zealand",    "IRLTLT01NZM156N", "^NZ50",   "NZ_NZX50"),
    "CA": ("Canada",         "IRLTLT01CAM156N", "^GSPTSE", "CA_TSX"),
}

# Dictionaries to store the resulting DataFrames
dev_10y_frames = {}
dev_eq_frames  = {}

for iso, (country, fred_id, eq_tk, eq_lbl) in other_dev_map.items():
    print(f"\n── {country} ──")

    # 1. Fetch 10-Year Yield
    try:
        dev_10y_frames[iso] = get_fred(fred_id, f"{iso}_10Y")
    except Exception as e:
        print(f"  ❌ FRED {fred_id}: {e}")

    # 2. Fetch Equity Index
    try:
        dev_eq_frames[iso] = get_yf(eq_tk, eq_lbl)
    except Exception as e:
        print(f"  ❌ YF {eq_tk}: {e}")

print("\n✅ Other Developed Markets 10Y and equity done.")


── United Kingdom ──
  ✅ FRED IRLTLT01GBM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^FTSE: 4796 obs | 2005-01-04 → 2023-12-29

── Switzerland ──
  ✅ FRED IRLTLT01CHM156N: 229 obs | 2005-01-01 → 2024-01-01


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^OBX']: YFPricesMissingError('possibly delisted; no price data found  (1d 2005-01-01 -> 2024-01-01)')


  ✅ YF ^SSMI: 4771 obs | 2005-01-03 → 2023-12-29

── Sweden ──
  ✅ FRED IRLTLT01SEM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^OMX: 3791 obs | 2008-11-20 → 2023-12-29

── Norway ──
  ✅ FRED IRLTLT01NOM156N: 229 obs | 2005-01-01 → 2024-01-01
  ⚠️  Yahoo Finance: no data for ^OBX

── Denmark ──
  ✅ FRED IRLTLT01DKM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^OMXC25: 1712 obs | 2016-12-19 → 2023-12-29

── Japan ──
  ✅ FRED IRLTLT01JPM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^N225: 4650 obs | 2005-01-04 → 2023-12-29

── Australia ──
  ✅ FRED IRLTLT01AUM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^AXJO: 4794 obs | 2005-01-04 → 2023-12-29

── New Zealand ──
  ✅ FRED IRLTLT01NZM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^NZ50: 4678 obs | 2005-01-05 → 2023-12-29

── Canada ──
  ✅ FRED IRLTLT01CAM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^GSPTSE: 4768 obs | 2005-01-04 → 2023-12-29

✅ Other Developed Markets 10Y and equity done.


In [45]:
# ## Cell 9 — Emerging Markets (FRED OECD + Yahoo Finance Equity)
# Pulls monthly 10-Year baseline yields from FRED and daily equity from YF.
# Note: Known dead/delisted endpoints have been set to 'None' to ensure error-free execution.

em_map = {
    # ISO: (Country, Bank, FRED_10Y, YF_Equity_Ticker, Equity_Label)
    "BR": ("Brazil",       "Central Bank of Brazil",    None,              "^BVSP",      "BR_Bovespa"),
    "MX": ("Mexico",       "Bank of Mexico",            "IRLTLT01MXM156N", "^MXX",       "MX_IPC"),
    "CL": ("Chile",        "Central Bank of Chile",     "IRLTLT01CLM156N", "^IPSA",      "CL_IPSA"),
    "IN": ("India",        "Reserve Bank of India",     "INDIRLTLT01STM",  "^NSEI",      "IN_Nifty50"),
    "KR": ("South Korea",  "Bank of Korea",             "IRLTLT01KRM156N", "^KS11",      "KR_KOSPI"),
    "ZA": ("South Africa", "South African Reserve Bank","IRLTLT01ZAM156N", "^J203.JO",   "ZA_JSE"),
    "IL": ("Israel",       "Bank of Israel",            "IRLTLT01ILM156N", "^TA125.TA",  "IL_TA125"),
    "HU": ("Hungary",      "Central Bank of Hungary",   "IRLTLT01HUM156N", "^BUX",       "HU_BUX"),
    "PL": ("Poland",       "National Bank of Poland",   "IRLTLT01PLM156N", None,         "PL_WIG20"),
    "CZ": ("Czech Rep.",   "Czech National Bank",       "IRLTLT01CZM156N", None,         "CZ_PX"),
}

em_10y_frames = {}
em_eq_frames  = {}

for iso, (country, bank, fred_id, eq_tk, eq_lbl) in em_map.items():
    print(f"\n── {country} ──")

    # 1. Fetch FRED 10Y Yield
    if fred_id:
        try:
            em_10y_frames[iso] = get_fred(fred_id, f"{iso}_10Y")
        except Exception as e:
            print(f"  ❌ FRED {fred_id}: {e}")
    else:
        print("  ℹ️  FRED 10Y: Data permanently deprecated by source.")

    # 2. Fetch Yahoo Finance Equity
    if eq_tk:
        try:
            em_eq_frames[iso]  = get_yf(eq_tk, eq_lbl)
        except Exception as e:
            print(f"  ❌ YF {eq_tk}: {e}")
    else:
        print("  ℹ️  YF Equity: Ticker permanently delisted.")

print("\n✅ Emerging market 10Y and equity done.")


── Brazil ──
  ℹ️  FRED 10Y: Data permanently deprecated by source.
  ✅ YF ^BVSP: 4699 obs | 2005-01-03 → 2023-12-28

── Mexico ──
  ✅ FRED IRLTLT01MXM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^MXX: 4767 obs | 2005-01-03 → 2023-12-29

── Chile ──
  ✅ FRED IRLTLT01CLM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^IPSA: 3595 obs | 2005-01-03 → 2019-06-14

── India ──
  ✅ FRED INDIRLTLT01STM: 146 obs | 2011-12-01 → 2024-01-01
  ✅ YF ^NSEI: 3992 obs | 2007-09-17 → 2023-12-29

── South Korea ──
  ✅ FRED IRLTLT01KRM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^KS11: 4690 obs | 2005-01-03 → 2023-12-28

── South Africa ──
  ✅ FRED IRLTLT01ZAM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^J203.JO: 2992 obs | 2012-02-08 → 2023-12-29

── Israel ──
  ✅ FRED IRLTLT01ILM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^TA125.TA: 4202 obs | 2005-01-03 → 2023-12-28

── Hungary ──
  ✅ FRED IRLTLT01HUM156N: 229 obs | 2005-01-01 → 2024-01-01
  ✅ YF ^BUX: 4091 obs | 2005-01-03 → 2021-09-30

── Pol

In [46]:
# ## Cell 10 — Asia-Pacific Equity Indices (Yahoo Finance)
# Pulls daily equity benchmarks for the APAC region.
# Note: Pakistan (^KSE100) has a known yfinance timezone bug and is set to None.

apac_eq = {
    "^N225":    "JP_Nikkei225",
    "^AXJO":    "AU_ASX200",
    "^NZ50":    "NZ_NZX50",
    "^STI":     "SG_STI",
    "^HSI":     "HK_HangSeng",
    "^KLSE":    "MY_KLCI",
    "^JKSE":    "ID_IDX",
    "^SET.BK":  "TH_SET",
    "PSEI.PS":  "PH_PSEi",
    "^KS11":    "KR_KOSPI",
    "000001.SS":"CN_SSEComposite",
    "^NSEI":    "IN_Nifty50",
    None:       "PK_KSE100",    # Bypassed to prevent YFTzMissingError crash
    "^OMXH25":  "FI_OMX25",
}

apac_eq_data = {}

print("── Asia-Pacific Equity Indices ──")
for tk, lbl in apac_eq.items():
    if tk:
        try:
            apac_eq_data[tk] = get_yf(tk, lbl)
        except Exception as e:
            print(f"  ❌ YF {tk}: {e}")
    else:
        print(f"  ℹ️  YF Equity [{lbl}]: Ticker bugged/delisted. Bypassed.")

print("\n✅ Asia-Pacific equity indices done.")

── Asia-Pacific Equity Indices ──
  ✅ YF ^N225: 4650 obs | 2005-01-04 → 2023-12-29
  ✅ YF ^AXJO: 4794 obs | 2005-01-04 → 2023-12-29
  ✅ YF ^NZ50: 4678 obs | 2005-01-05 → 2023-12-29
  ✅ YF ^STI: 4747 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^HSI: 4680 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^KLSE: 4660 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^JKSE: 4619 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^SET.BK: 4625 obs | 2005-01-04 → 2023-12-28
  ✅ YF PSEI.PS: 4663 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^KS11: 4690 obs | 2005-01-03 → 2023-12-28
  ✅ YF 000001.SS: 4612 obs | 2005-01-04 → 2023-12-29
  ✅ YF ^NSEI: 3992 obs | 2007-09-17 → 2023-12-29
  ℹ️  YF Equity [PK_KSE100]: Ticker bugged/delisted. Bypassed.
  ✅ YF ^OMXH25: 2708 obs | 2013-03-05 → 2023-12-29

✅ Asia-Pacific equity indices done.


In [47]:
# ## Cell 11 — Latin America, EMEA Equity Indices (Yahoo Finance)
# Pulls global equity benchmarks.
# Note: Known dead/delisted/bugged YF endpoints are set to 'None' to prevent crashes.

other_eq = {
    # ── Latin America ──
    "^BVSP":      "BR_Bovespa",
    "^MXX":       "MX_IPC",
    "^IPSA":      "CL_IPSA",
    None:         "CO_COLCAP",     # Bypassed (YFTzMissingError)
    "^MERV":      "AR_Merval",
    None:         "PE_BVL",        # Bypassed (Delisted)
    "^GSPTSE":    "CA_TSX",

    # ── Middle East & Africa ──
    "^TASI.SR":   "SA_TASI",
    None:         "UAE_ADX",       # Bypassed (YFTzMissingError)
    None:         "TR_BIST100",    # Bypassed (Delisted)
    "^TA125.TA":  "IL_TA125",
    None:         "MA_MASI",       # Bypassed (Rate Limited/Bugged)
    "^J203.JO":   "ZA_JSE",

    # ── Emerging Europe ──
    "^OMXTGI":    "EE_OMXTallinn",
    "^OMXRGI":    "LV_OMXRiga",
    "^OMXVGI":    "LT_OMXVilnius",
    "^BUX":       "HU_BUX",
    None:         "RO_BET",        # Bypassed (YFTzMissingError)
    None:         "BG_SOFIX",      # Bypassed (Delisted)
    None:         "HR_CROBEX",     # Bypassed (YFTzMissingError)
    None:         "CZ_PX",         # Bypassed (Delisted)
    None:         "PL_WIG20",      # Bypassed (Delisted)
    "IMOEX.ME":   "RU_MOEX",

    # ── Major Europe (Cross-validation) ──
    "^ATX":       "AT_ATX",
    "^BFX":       "BE_BEL20",
    "^AEX":       "NL_AEX",
    None:         "PT_PSI20",      # Bypassed (Delisted)
    "^ISEQ":      "IE_ISEQ",
    "GD.AT":      "GR_ASE",
    "FTSEMIB.MI": "IT_FTSEMIB",
    "^IBEX":      "ES_IBEX35",
    "^FCHI":      "FR_CAC40",
    "^GDAXI":     "DE_DAX",
    "^FTSE":      "UK_FTSE100",
    "^SSMI":      "CH_SMI",
    "^OMX":       "SE_OMX30",
    None:         "NO_OBX",        # Bypassed (Delisted)
    "^OMXC25":    "DK_OMXC25",
    "^OMXIPI":    "IS_OMXI",
    "^STOXX50E":  "EA_EuroStoxx50",
}

other_eq_data = {}

print("── Latin America & EMEA Equity Indices ──")
for tk, lbl in other_eq.items():
    if tk:
        try:
            other_eq_data[tk] = get_yf(tk, lbl)
        except Exception as e:
            print(f"  ❌ YF {tk}: {e}")
    else:
        print(f"  ℹ️  YF Equity [{lbl}]: Ticker bugged/delisted. Bypassed.")

print("\n✅ LatAm + EMEA equity done.")

── Latin America & EMEA Equity Indices ──
  ✅ YF ^BVSP: 4699 obs | 2005-01-03 → 2023-12-28
  ✅ YF ^MXX: 4767 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^IPSA: 3595 obs | 2005-01-03 → 2019-06-14
  ℹ️  YF Equity [NO_OBX]: Ticker bugged/delisted. Bypassed.
  ✅ YF ^MERV: 4633 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^GSPTSE: 4768 obs | 2005-01-04 → 2023-12-29
  ✅ YF ^TASI.SR: 4098 obs | 2005-01-03 → 2023-12-31
  ✅ YF ^TA125.TA: 4202 obs | 2005-01-03 → 2023-12-28
  ✅ YF ^J203.JO: 2992 obs | 2012-02-08 → 2023-12-29
  ✅ YF ^OMXTGI: 2642 obs | 2013-04-30 → 2023-12-29
  ✅ YF ^OMXRGI: 1184 obs | 2019-02-08 → 2023-12-29
  ✅ YF ^OMXVGI: 2634 obs | 2013-04-30 → 2023-12-29
  ✅ YF ^BUX: 4091 obs | 2005-01-03 → 2021-09-30
  ✅ YF IMOEX.ME: 2682 obs | 2013-03-05 → 2023-12-29
  ✅ YF ^ATX: 4722 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^BFX: 4859 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^AEX: 4862 obs | 2005-01-03 → 2023-12-29
  ✅ YF ^ISEQ: 4820 obs | 2005-01-04 → 2023-12-29
  ✅ YF GD.AT: 4704 obs | 2005-01-03 → 2023-12-29


In [48]:
# ## Cell 12 — Additional OECD 10Y Yields via FRED
# Remaining emerging and frontier markets.
# Note: FRED has permanently deprecated many of these specific proxy series.
# We bypass the dead endpoints (None) to guarantee an error-free execution.

extra_oecd = {
    None:              "SG_10Y_monthly",   # Bypassed (404)
    None:              "ID_10Y_monthly",   # Bypassed (404)
    None:              "TR_10Y_monthly",   # Bypassed (404)
    None:              "IN_10Y_monthly",   # Bypassed (404)
    None:              "CN_10Y_monthly",   # Bypassed (404)
    None:              "MY_10Y_monthly",   # Bypassed (404)
    None:              "PH_10Y_monthly",   # Bypassed (404)
    None:              "TH_10Y_monthly",   # Bypassed (404)
    None:              "HK_10Y_monthly",   # Bypassed (404)
    "IRLTLT01RUM156N": "RU_10Y_monthly",   # Russia (Active)
}

extra_10y = {}

print("── Additional OECD 10Y Yields ──")
for series, lbl in extra_oecd.items():
    if series:
        try:
            extra_10y[lbl] = get_fred(series, lbl)
        except Exception as e:
            print(f"  ❌ FRED {series}: {e}")
    else:
        print(f"  ℹ️  FRED 10Y [{lbl}]: Series deprecated by source. Bypassed.")

print("\n✅ Extra OECD 10Y done.")

── Additional OECD 10Y Yields ──
  ℹ️  FRED 10Y [HK_10Y_monthly]: Series deprecated by source. Bypassed.
  ✅ FRED IRLTLT01RUM156N: 162 obs | 2005-01-01 → 2018-06-01

✅ Extra OECD 10Y done.


In [49]:
# ## Cell 13 — Master Dataset Merge, Validation, and Export
# This cell compiles all collected data, aligns the dates, and exports to CSV.

import pandas as pd

print("── Compiling Master Dataset ──")

all_frames = []

def safe_append(df):
    """Safely append a dataframe to the master list if it contains data."""
    if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
        # Ensure the index is formatted uniformly as Datetime
        if df.index.name != 'Date':
            df.index.name = 'Date'
        df.index = pd.to_datetime(df.index)
        all_frames.append(df)

def safe_append_dict(df_dict):
    """Unpack dictionaries of dataframes safely."""
    if isinstance(df_dict, dict):
        for _, df in df_dict.items():
            safe_append(df)

# 1. Append Individual Core DataFrames (US & ECB)
safe_append(us_df)
safe_append(ecb_2y)
safe_append(ecb_10y)

# 2. Append the batch loops (Eurozone, Developed, Emerging, LatAm, etc.)
safe_append_dict(ez_10y_frames)
safe_append_dict(ez_eq_frames)
safe_append_dict(dev_10y_frames)
safe_append_dict(dev_eq_frames)
safe_append_dict(stooq_2y_data)  # Safely processes the empty bypass dict
safe_append_dict(stooq_10y_data) # Safely processes the empty bypass dict
safe_append_dict(em_10y_frames)
safe_append_dict(em_eq_frames)
safe_append_dict(apac_eq_data)
safe_append_dict(other_eq_data)
safe_append_dict(extra_10y)

# 3. Concatenate everything along the Date index
master_df = pd.concat(all_frames, axis=1)

# 4. Clean up duplicate columns (if any overlap occurred)
master_df = master_df.loc[:, ~master_df.columns.duplicated()]

# 5. Sort chronologically and restrict strictly to the study window
master_df = master_df.sort_index()
master_df = master_df.loc[START:END]

print(f"✅ Master DataFrame compiled: {master_df.shape[0]} rows (trading days) × {master_df.shape[1]} columns (variables)")

# 6. Export the final dataset
dataset_filename = "global_cb_financial_data_2005_2023.csv"
master_df.to_csv(dataset_filename)
print(f"✅ Dataset saved to Colab files: {dataset_filename}")

# 7. Generate and export the Coverage Report
coverage = master_df.notna().sum().sort_values(ascending=False).to_frame(name="Valid_Observations")
coverage["Coverage_%"] = (coverage["Valid_Observations"] / len(master_df) * 100).round(1)
coverage.to_csv("coverage_report.csv")
print("✅ Coverage report saved to Colab files: coverage_report.csv")

# Display the top 10 most complete variables
print("\n── Top 10 Variables by Coverage ──")
print(coverage.head(10))

── Compiling Master Dataset ──
✅ Master DataFrame compiled: 5276 rows (trading days) × 76 columns (variables)
✅ Dataset saved to Colab files: global_cb_financial_data_2005_2023.csv
✅ Coverage report saved to Colab files: coverage_report.csv

── Top 10 Variables by Coverage ──
            Valid_Observations  Coverage_%
NL_AEX                    4862        92.2
FR_CAC40                  4859        92.1
BE_BEL20                  4859        92.1
EA_2Y_ECB                 4856        92.0
EA_10Y_ECB                4856        92.0
ES_IBEX35                 4852        92.0
IT_FTSEMIB                4830        91.5
DE_DAX                    4824        91.4
IE_ISEQ                   4820        91.4
UK_FTSE100                4796        90.9
